# Test Query Team Execution

This notebook demonstrates two ways to run the query team workflow:
1. **Using QueryManager**: Simulates the standard way of submitting a query and getting the final result.
2. **Using Direct Graph Stream**: Directly interacts with the LangGraph instance to observe intermediate steps.

## 七个模型综合总结与评分

本次评测考察了七个大型语言模型（A-G）在处理一系列关于基于指示剂置换分析（IDA）的奎宁电化学传感器的问答对时的表现。模型包括不同版本（gpt-4o, gpt-4.1, o4-mini, nano）以及不同运行模式（直接回答 vs. 查询增强后回答）。评测核心关注准确性、上下文理解、技术深度、结构和对目标受众（超分子化学研究者）的适切性。

**核心发现:**

1.  **IDA 术语理解是关键:** 模型的成败很大程度上取决于是否能正确理解核心术语 "IDA" 在此上下文（Indicator Displacement Assay）中的含义。只有模型 **A** 和 **C** 做到了这一点。模型 **B** 将其误解为碘乙酰胺/阻抗分析。模型 **D, E, F, G** 则一致将其误解为 Interdigitated Array (叉指阵列) 电极。这个基础性错误导致后者在关键问题（Q5, Q8）上的回答完全偏离主题，尽管它们在其他问题上可能表现出深度。
2.  **上下文感知能力差异悬殊:** 模型 **C** 在理解和利用问题背后隐含的特定传感器系统信息（β-CD 主体、具体性能指标等）方面表现突出，其答案与用户基准高度一致。模型 **A** 也展现了较好的上下文感知能力。其他模型则普遍缺乏这种能力，倾向于提供通用答案，或在错误的前提下进行推演。
3.  **查询增强 (`query + model`) 模式优势显著:** 对比 `query + nano` 模型（A, C）和 `direct answer` 模型（B, D, E, F, G），前者在理解术语、把握上下文方面表现明显更优。查询步骤似乎有效地引导模型聚焦于正确的领域和细节。
4.  **细节深度 vs. 核心准确性:** 模型 **F** 和 **G** (均为 o4-mini) 在提供极其详尽的技术细节、良好结构和广泛覆盖方面表现突出，甚至超过了 gpt-4.1 (D)。然而，它们在核心术语上的错误使得这些细节在关键问题上失去了意义，甚至可能产生误导。这表明，对于专业领域任务，核心概念的准确性优先于信息的堆砌。
5.  **模型大小与表现并非完全正相关:** 虽然 nano 模型 (B) 表现最差，但 o4-mini (F, G) 和 gpt-4.1 (D), gpt-4o (E) 这些更大的模型在关键的术语理解上同样失败。反而是在查询增强模式下，gpt-4.1 (C) 和 gpt-4o (A) (驱动 nano 回答) 表现更好。这凸显了运行模式和引导的重要性。

**各模型表现总结:**

* **模型 A (gpt-4o 查询 + nano):** 表现良好。正确理解 IDA，上下文感知较好，能提供具体数据。主要缺点是在 Q5 的结论（认为不存在此类传感器）与特定上下文相悖。
* **模型 B (nano 直接回答):** 表现最差。未能正确理解 IDA，回答普遍笼统、缺乏深度，是不可靠的选择。
* **模型 C (gpt-4.1 查询 + nano):** **表现最佳**。准确理解 IDA，上下文感知能力极强，答案精准且与用户基准高度吻合，完美契合任务需求。唯一的遗憾是缺失了 Q2 的回答。
* **模型 D (gpt-4.1 直接回答):** 表现不佳。虽然通用知识性问题回答结构尚可，并包含引用，但未能正确理解 IDA，且缺乏对特定上下文的把握。
* **模型 E (gpt-4o 直接回答):** 表现不佳。与 D 类似，未能正确理解 IDA，缺乏上下文感知，且细节和结构不如 D/F/G。
* **模型 F (o4-mini 直接回答):** 细节极其丰富，结构良好。但在关键的 IDA 理解上失败，且在 Q5 基于错误前提提供了看似详细实则误导的信息和引用。优点被核心错误掩盖。
* **模型 G (o4-mini 直接回答, high cost):** 细节极为丰富，结构好，通用知识问题回答出色。但在关键的 IDA 理解上同样失败。虽然 Q7 提供的“典型”数据非常接近真实值，显示出一定的潜力，但核心概念错误使其无法胜任此任务。

---

**最终评分与原因 (1-10分):**

1.  **模型 C (gpt-4.1 查询 + nano): 9.5 / 10**
    * **原因:** IDA 理解正确，上下文感知极佳，答案准确具体，与基准高度吻合。近乎完美，仅因缺失 Q2 略作扣分。
2.  **模型 A (gpt-4o 查询 + nano): 7.5 / 10**
    * **原因:** IDA 理解正确，上下文感知较好，提供了部分具体数据。但在关键问题 Q5 上结论与特定上下文冲突。
3.  **模型 G (o4-mini direct, high cost): 5.0 / 10**
    * **原因:** 通用知识问题极为详尽，结构好。但 IDA 理解错误（关键缺陷），导致核心问题回答错误，高成本未解决核心问题。细节丰富度使其略高于犯同样错误的 D/E/F。
4.  **模型 F (o4-mini direct): 4.5 / 10**
    * **原因:** 通用知识问题非常详尽，结构好。但 IDA 理解错误（关键缺陷），且 Q5 的误导性细节和引用问题严重。
5.  **模型 D (gpt-4.1 direct): 4.0 / 10**
    * **原因:** 结构和引用尚可。但 IDA 理解错误（关键缺陷），上下文感知差，未能提供特定数据。
6.  **模型 E (gpt-4o direct): 3.5 / 10**
    * **原因:** IDA 理解错误（关键缺陷），上下文感知差，细节和结构不如 D/F/G。
7.  **模型 B (nano direct): 2.0 / 10**
    * **原因:** IDA 理解错误（关键缺陷），回答普遍笼统，缺乏深度和准确性，基本不可用。

## Part 0: Setup (Imports and Ontology)

In [1]:
import sys
import os
sys.path.append(r"D:\\CursorProj\\Chem-Ontology-Constructor")
os.environ["PROJECT_ROOT"] = "D:\\\\CursorProj\\\\Chem-Ontology-Constructor\\\\"

from owlready2 import get_ontology
# 从 config.settings 导入 ONTOLOGY_SETTINGS 而不是 ONTOLOGY_CONFIG
from config.settings import ONTOLOGY_SETTINGS
# 本体现在在 ONTOLOGY_SETTINGS 初始化时加载，如果需要，可以通过 ONTOLOGY_SETTINGS.ontology 访问
# 例如: onto = ONTOLOGY_SETTINGS.ontology
# onto_additional = get_ontology("data/ontology/test.owl").load() # 可选的第二个本体

Setting owlready2.JAVA_EXE globally from settings.yaml: C:\Program Files\Java\jdk-23\bin\java.exe


In [10]:
from langchain_openai import ChatOpenAI

answer_llm = ChatOpenAI(
            model_name="o4-mini",
            # temperature=0,
            max_tokens=10000,
            reasoning_effort="high"
        )

In [3]:
# Required Imports
import sys
import os
import json
import time
from typing import Dict, Any, List
from owlready2 import *
import asyncio # Needed for owlready2 async operations in some envs

# Import the OntologySettings class
from config.settings import OntologySettings, ONTOLOGY_SETTINGS # Keep ONTOLOGY_SETTINGS import for potential base_iri access

# Import necessary LLM and Query Team components
try:
    from autology_constructor.idea.query_team import QueryManager, Query, QueryStatus, create_query_graph
    from autology_constructor.idea.query_team.ontology_tools import OntologyTools
    from autology_constructor.idea.common.llm_provider import get_cached_default_llm
    print("Modules imported successfully.")
except ModuleNotFoundError as e:
    print(f"Error importing modules: {e}")
    print(f"Current sys.path: {sys.path}")

# Ensure LLM Provider is configured
try:
    llm = get_cached_default_llm()
    print(f"LLM: {llm.model_name}. \nAnsewr LLM: {answer_llm.model_name}")
    print("LLM Provider initialized successfully.")
except Exception as e:
    print(f"Error initializing LLM Provider: {e}\nPlease ensure API keys or necessary configurations are set.")
    llm = None

# --- Ontology Setup ---
print("Setting up test ontology using a new OntologySettings instance...")

# Define parameters for the new OntologySettings instance
# Assuming 'backup-2.owl' and 'backup-2-closed.owl' exist in 'data/ontology'
# Use the project root defined in the previous cell
project_root_path = os.environ.get("PROJECT_ROOT", ".")
ontology_dir = os.path.join(project_root_path, "data", "ontology")
# You might want to use the base_iri from the default settings or define a specific one for testing
test_base_iri = ONTOLOGY_SETTINGS.base_iri if 'ONTOLOGY_SETTINGS' in locals() else "http://www.test.org/chem_ontologies/backup-2"

try:
    # Instantiate OntologySettings directly
    # test_ontology_settings = ONTOLOGY_SETTINGS
    test_ontology_settings = OntologySettings(
        base_iri=test_base_iri,
        ontology_file_name="IDA.owl",  # Use the desired ontology file
        directory_path=ontology_dir,
        closed_ontology_file_name="IDA-closed.owl" # Adjust if your closed file has a different name pattern
    )
    # Access the loaded ontology via the instance's property
    test_onto = test_ontology_settings.ontology
    print(f"Successfully loaded ontology: {test_onto.base_iri}")
    print(f"From file: {test_ontology_settings.ontology_file_name} in {test_ontology_settings.directory_path}")

    # Optional: Print some details about the loaded ontology
    # print(f"Test Ontology '{test_onto.base_iri}' loaded with:")
    # print(f"- Classes ({len(list(test_onto.classes()))}): {[c.name for c in list(test_onto.classes())[:5]]}...") # Print first 5
    # print(f"- Individuals ({len(list(test_onto.individuals()))}): {[i.name for i in list(test_onto.individuals())[:5]]}...")
    # print(f"- Object Properties ({len(list(test_onto.object_properties()))}): {[p.name for p in list(test_onto.object_properties())[:5]]}...")
    # print(f"- Data Properties ({len(list(test_onto.data_properties()))}): {[p.name for p in list(test_onto.data_properties())[:5]]}...")

except Exception as e:
    print(f"Error creating OntologySettings or loading ontology 'backup-2.owl': {e}")
    print(f"Please ensure 'backup-2.owl' exists in '{ontology_dir}' and settings are correct.")
    test_onto = None # Set to None if loading failed

# # --- Old way (commented out) ---
# # print("Creating a simple in-memory ontology...")
# # # It's good practice to clear existing ontologies from the default world if running cells repeatedly
# # for o in list(default_world.ontologies.values()):
# #     if callable(getattr(o, '__destroy__', None)):
# #         try:
# #             destroy_entity(o)
# #         except Exception as destroy_err:
# #             print(f"Error destroying {o.base_iri}: {destroy_err}")
# #     else:
# #         print(f"Skipping destroy for non-callable __destroy__ or missing: {o.base_iri}")
# # test_onto_old = get_ontology("data/ontology/test.owl").load()
# # for o in list(default_world.ontologies.values()):
# #     print(f"has：{o.base_iri}")
# # print(f"Test Ontology '{test_onto_old.base_iri}' created with:\\n- Classes: {[c.name for c in test_onto_old.classes()]}\\n- Individuals: {[i.name for i in test_onto_old.individuals()]}\\n- Object Properties: {[p.name for p in test_onto_old.object_properties()]}\\n- Data Properties: {[p.name for p in test_onto_old.data_properties()]}\")

# Run async tasks if needed by owlready2 backend (usually not necessary for simple loading)
# try:
#     loop = asyncio.get_event_loop()
# except RuntimeError:
#     loop = asyncio.new_event_loop()
#     asyncio.set_event_loop(loop)
# loop.run_until_complete(asyncio.sleep(0)) # Run pending async tasks

Modules imported successfully.
LLM: gpt-4.1. 
Ansewr LLM: o4-mini
LLM Provider initialized successfully.
Setting up test ontology using a new OntologySettings instance...
Successfully loaded ontology: http://www.test.org/chem_ontologies/chem_ontology.owl#
From file: IDA.owl in D:\\CursorProj\\Chem-Ontology-Constructor\\data\ontology


## Part 1: Execution via QueryManager

In [4]:
qas = {
  "query_format_QA": [
    {
      "difficulty_level": 1,
      "query": "Quinine definition?",
      "answer": "Quinine is an alkaloid derived from cinchona tree bark, used historically for malaria and now as a bittering agent, but associated with adverse health effects like thrombocytopenia."
    },
    {
      "difficulty_level": 1,
      "query": "Indicator Displacement Assay (IDA) definition?",
      "answer": "An IDA is a sensing strategy based on host-guest recognition, often utilizing non-covalent interactions where an analyte displaces an indicator from a receptor, causing a detectable signal change (e.g., fluorescence or absorbance)."
    },
    {
      "difficulty_level": 2,
      "query": "What analyzes Quinine?",
      "answer": "Quinine is analyzed by techniques including Electrochemical Technique, High Performance Liquid Chromatography (HPLC), Colorimetric Assay, Fluorescence Assay, and High Resolution Mass Spectrometry (HRMS)."
    },
    {
      "difficulty_level": 2,
      "query": "List components of Indicator Displacement Assay (IDA).",
      "answer": "Components include beta-Cyclodextrin (beta-CD), Poly(N-acetylaniline), and Graphene."
    },
    {
      "difficulty_level": 3,
      "query": "Find electrochemical sensors based on IDA used for Quinine detection.",
      "answer": "The Electrochemical Sensor constructed via IDA (Indicator Displacement Assay) has the detection target Quinine."
    },
    {
      "difficulty_level": 3,
      "query": "Which hosts use host-guest recognition and are integrated with electrochemical assays?",
      "answer": "Host-Guest Recognition is integrated with an Electrochemical Sensor. Beta-Cyclodextrin (β-CD) is a host involved in Host-Guest Recognition."
    },
    {
      "difficulty_level": 4,
      "query": "Compare the stability and reproducibility properties of the Electrochemical Sensor.",
      "answer": "The Electrochemical Sensor has high stability (acceptable peak current decrease within 21 days, 86.47% retained) and good reproducibility (RSD of 2.06% across seven electrodes)."
    },
    {
      "difficulty_level": 4,
      "query": "What techniques verify the Electrochemical Sensor based on IDA?",
      "answer": "The Electrochemical Sensor is verified by Differential Pulse Voltammetry (DPV), Cyclic Voltammetry (CV), Proton Nuclear Magnetic Resonance (H_NMR), Scanning Electron Microscopy (SEM), Electrochemical Impedance Analysis (EIS), and Fourier Transform Infrared (FTIR)."
    },
    {
      "difficulty_level": 5,
      "query": "Explain the sensing mechanism involving Methylene Blue (MB) displacement by Quinine from beta-Cyclodextrin (beta-CD).",
      "answer": "Methylene Blue (MB) forms an inclusion complex with beta-Cyclodextrin (beta-CD). Quinine, having a higher binding affinity, competitively displaces MB from the beta-CD cavity. This displacement causes a change in the electrochemical signal (e.g., DPV peak current) which is used for Quinine detection. Poly(N-acetylaniline) inhibits non-specific adsorption of MB, contributing to the assay's selectivity."
    },
    {
      "difficulty_level": 5,
      "query": "Summarize the role of Graphene in the described electrochemical sensor.",
      "answer": "Graphene (specifically reduced graphene oxide, rGO) is used as an electrode material in the sensor. It enhances electron transfer properties due to its superior electrical conductivity and large specific surface area, improving the sensor's performance. It serves as a platform onto which other components like Poly(N-acetylaniline) and beta-Cyclodextrin are deposited."
    }
  ],
  "question_format_QA": [
    {
      "difficulty_level": 1,
      "question": "Tell me about Quinine.",
      "answer": "Quinine is an alkaloid derived from cinchona tree bark, used historically for malaria and now as a bittering agent, but associated with adverse health effects like thrombocytopenia."
    },
    {
      "difficulty_level": 1,
      "question": "What is an Indicator Displacement Assay?",
      "answer": "An IDA is a sensing strategy based on host-guest recognition, often utilizing non-covalent interactions where an analyte displaces an indicator from a receptor, causing a detectable signal change (e.g., fluorescence or absorbance)."
    },
    {
      "difficulty_level": 2,
      "question": "What techniques are used to analyze Quinine?",
      "answer": "Quinine is analyzed by techniques including Electrochemical Technique, High Performance Liquid Chromatography (HPLC), Colorimetric Assay, Fluorescence Assay, and High Resolution Mass Spectrometry (HRMS)."
    },
    {
      "difficulty_level": 2,
      "question": "What are the components of an Indicator Displacement Assay?",
      "answer": "Components include beta-Cyclodextrin (beta-CD), Poly(N-acetylaniline), and Graphene."
    },
    {
      "difficulty_level": 3,
      "question": "Are there electrochemical sensors using IDA to detect Quinine?",
      "answer": "The Electrochemical Sensor constructed via IDA (Indicator Displacement Assay) has the detection target Quinine."
    },
    {
      "difficulty_level": 3,
      "question": "Which host molecules use host-guest recognition in electrochemical assays?",
      "answer": "Host-Guest Recognition is integrated with an Electrochemical Sensor. Beta-Cyclodextrin (β-CD) is a host involved in Host-Guest Recognition."
    },
    {
      "difficulty_level": 4,
      "question": "How stable and reproducible is the electrochemical sensor?",
      "answer": "The Electrochemical Sensor has high stability (acceptable peak current decrease within 21 days, 86.47% retained) and good reproducibility (RSD of 2.06% across seven electrodes)."
    },
    {
      "difficulty_level": 4,
      "question": "How is the IDA-based electrochemical sensor verified?",
      "answer": "The Electrochemical Sensor is verified by Differential Pulse Voltammetry (DPV), Cyclic Voltammetry (CV), Proton Nuclear Magnetic Resonance (H_NMR), Scanning Electron Microscopy (SEM), Electrochemical Impedance Analysis (EIS), and Fourier Transform Infrared (FTIR)."
    },
    {
      "difficulty_level": 5,
      "question": "How does Quinine displace Methylene Blue from beta-Cyclodextrin in the sensor?",
      "answer": "Methylene Blue (MB) forms an inclusion complex with beta-Cyclodextrin (beta-CD). Quinine, having a higher binding affinity, competitively displaces MB from the beta-CD cavity. This displacement causes a change in the electrochemical signal (e.g., DPV peak current) which is used for Quinine detection. Poly(N-acetylaniline) inhibits non-specific adsorption of MB, contributing to the assay's selectivity."
    },
    {
      "difficulty_level": 5,
      "question": "What does Graphene do in the electrochemical sensor?",
      "answer": "Graphene (specifically reduced graphene oxide, rGO) is used as an electrode material in the sensor. It enhances electron transfer properties due to its superior electrical conductivity and large specific surface area, improving the sensor's performance. It serves as a platform onto which other components like Poly(N-acetylaniline) and beta-Cyclodextrin are deposited."
    }
  ]
}

In [5]:
num = 10
# 定义新的十个查询
queries = [item["query"] for item in qas["query_format_QA"][:num]]

revised_queries = [item["question"] for item in qas["question_format_QA"][:num]]

# 为所有查询定义统一的上下文
query_context = {
    "ontology": test_ontology_settings,
    "originating_team": "test_notebook",
    "originating_stage": "manual_test",
    "query_type": "information_retrieval" # 对所有查询使用信息检索类型
}

print(len(queries),len(revised_queries))

10 10


In [6]:
if not llm:
    print("Skipping QueryManager test due to LLM initialization failure.")
else:
    print("--- Starting QueryManager Test ---")
    query_manager = QueryManager()

    # 更新类缓存
    print("Updating class name cache...")
    query_manager.update_class_name_cache(test_onto)
    # 打印部分缓存内容以确认
    if query_manager.class_name_cache:
         print(f"Cache content (first 10): {query_manager.class_name_cache[:10]}...")
    else:
         print("Class name cache is empty.")


    # 启动管理器
    print("Starting QueryManager...")
    query_manager.start()

    # 提交多个查询并收集futures
    futures = []
    print(f"\\nSubmitting {len(queries)} queries...")
    for i, query_text in enumerate(queries):
        print(f"Submitting query {i+1}: '{query_text[:80]}...'") # 打印部分查询文本
        future = query_manager.submit_query(query_text=query_text, query_context=query_context)
        futures.append((i+1, query_text, future))

    print("\nAll queries submitted.")

--- Starting QueryManager Test ---
Updating class name cache...
Class name cache updated with 351 classes.
Cache content (first 10): ['1,3,5-triethyl-2,4,6-trimethylamine', '1,3-diynyl', '1,4-triazole', '1:1_inclusion_complexes', '1H_NMR_spectroscopic_titration', '1H_NMR_spectroscopic_titrations', '1H_NMR_spectrum', '1_palmitoyl_2_oleoyl_sn_glycero_3_phosphocholine(POPC)', '23a', '25⊂(24)2']...
Starting QueryManager...
Dispatcher loop started on thread QueryDispatcherThread
Query Manager dispatcher started.
\nSubmitting 10 queries...
Submitting query 1: 'What description is provided for calix[4]pyrrole in the ontology?...'
Submitting query 2: 'What is the pKa value mentioned for methanesulfonic acid (MSA)?...'
Submitting query 3: 'What type of reaction is used to synthesize aryl-extended calix(4)pyrroles (AE-C...'
Submitting query 4: 'List some specific types (subclasses) of macrocyclic receptors mentioned in the ...'
Submitting query 5: 'According to the ontology, what can enhance the

http://www.test.org/chem_ontologies/meta/
http://www.test.org/chem_ontologies/classes/
http://www.test.org/chem_ontologies/object_properties/
http://www.test.org/chem_ontologies/data_properties/
http://www.test.org/chem_ontologies/meta/
http://www.test.org/chem_ontologies/classes/
http://www.test.org/chem_ontologies/object_properties/
http://www.test.org/chem_ontologies/data_properties/
http://www.test.org/chem_ontologies/meta/
http://www.test.org/chem_ontologies/classes/
http://www.test.org/chem_ontologies/object_properties/
http://www.test.org/chem_ontologies/data_properties/
http://www.test.org/chem_ontologies/meta/
http://www.test.org/chem_ontologies/classes/
http://www.test.org/chem_ontologies/object_properties/
http://www.test.org/chem_ontologies/data_properties/
{
  "results": [
    {
      "tool": "parse_class_definition",
      "params": {
        "class_names": [
          "calix(4)pyrrole"
        ]
      },
      "result": {
        "calix(4)pyrrole": {
          "basic_inf

In [7]:
if not llm:
    print("Skipping QueryManager test due to LLM initialization failure.")
else:    
    # 等待并获取所有结果
    print("Waiting for queries completion...")
    try:
        for i, query_text, future in futures:
            print(f"\nProcessing results for query {i}: '{query_text}'")
            # 等待合理的时间（根据需要调整）
            final_result_dict = future.result(timeout=120)
            print(f"Query {i} completed.")
            # 美观打印最终状态字典
            print(f"\n--- Final State Dictionary for Query {i} ---")
            # 使用default=str处理潜在的不可序列化对象，如本体引用
            print(json.dumps(final_result_dict, indent=2, default=str))
    except Exception as e:
        print(f"Error getting query result: {e}")
        if future.done() and future.exception():
             print(f"Future exception details: {future.exception()}")
    finally:
        # 停止管理器
        print("\nStopping QueryManager...")
        query_manager.stop()
        print("QueryManager stopped.")

    print("--- QueryManager Test Finished ---")


Waiting for queries completion...

Processing results for query 1: 'What description is provided for calix[4]pyrrole in the ontology?'
Query 1 completed.

--- Final State Dictionary for Query 1 ---
{
  "query": "What description is provided for calix[4]pyrrole in the ontology?",
  "source_ontology": "OntologySettings(base_iri='http://www.test.org/chem_ontologies/', ontology_file_name='backup-2.owl', directory_path='D:\\\\\\\\CursorProj\\\\\\\\Chem-Ontology-Constructor\\\\\\\\data\\\\ontology', closed_ontology_file_name='backup-2-closed.owl')",
  "query_type": "information_retrieval",
  "query_strategy": "tool_sequence",
  "originating_team": "test_notebook",
  "originating_stage": "manual_test",
  "available_classes": [
    "1,3,5-triethyl-2,4,6-trimethylamine",
    "1,3-diynyl",
    "1,4-triazole",
    "1:1_inclusion_complexes",
    "1H_NMR_spectroscopic_titration",
    "1H_NMR_spectroscopic_titrations",
    "1H_NMR_spectrum",
    "1_palmitoyl_2_oleoyl_sn_glycero_3_phosphocholine(POPC

In [6]:
# 定义回调函数处理Future结果并使用agent生成回答

def process_result_with_agent(result_dict, query_text):
    """
    Use an agent to process query results and generate a natural language response
    
    Args:
        result_dict: Query result dictionary
        query_text: Original query text
    
    Returns:
        str: Natural language response generated by the agent
    """
    # Extract query results information from the result
    if "formatted_results" in result_dict:
        query_results = result_dict["formatted_results"]
        print("-"*100)
        print(query_results)
    else:
        return f"I'm sorry, I couldn't find valid information about '{query_text}'."
    
    # Construct the prompt in English
    prompt = f"""
**Role:** You are an expert Chemistry Researcher.

**Task:** Provide a clear, accurate, and comprehensive answer to the user's question. You should leverage your own expert knowledge, **judiciously enhancing and verifying** it with **relevant and applicable information** selected from the 'Ontology query results'.

**User Question:**
{query_text}

**Information Source (Ontology Query Results for Enhancement & Verification):**
{query_results}

**Response Guidelines:**
* **Knowledge Integration:** Synthesize your broad chemical knowledge with **pertinent details** from the 'Information Source'.
* **Selective Use of Source:** Critically evaluate the 'Information Source'. **Incorporate specific details** (e.g., data points like pKa values, reaction types, precise definitions, specific examples) **only when they directly enhance the accuracy, specificity, or completeness of the answer to the user's question.** Do not feel obligated to include all provided information; prioritize relevance to the query.
* **Verification and Conflict:** Use the source to verify facts where appropriate. If there's a conflict between your general knowledge and the source, prioritize the source's specific data **if it is relevant to the question and appears accurate**, but use your expert judgment to omit information that seems erroneous or irrelevant to the user's query.
* **Synthesis:** Weave together your general knowledge and the selected source information into a coherent, well-structured response.
* **Clarity & Tone:** Use precise, professional chemical language. Aim for accessibility by briefly explaining potentially niche terms if needed.
* **Directness & Comprehensiveness:** Address all parts of the user's question directly and thoroughly, enriched by the appropriately selected information.
* **Source Attribution:** Do **not** mention "ontology" or refer to the 'Information Source' explicitly (e.g., avoid "according to the provided data..."). Present the integrated information as established chemical facts.

**Answer:**
"""
    
    # Generate response using LLM
    try:
        response = answer_llm.invoke(prompt)
        return response
    except Exception as e:
        return f"Error generating response: {e}"

def query_result_callback(future, query_idx, query_text):
    """Callback function to process Future results"""
    try:
        print(f"\nProcessing callback for query {query_idx}: '{query_text}'")
        
        # Get the future result
        result_dict = future.result(timeout=5)  # Small timeout to avoid indefinite waiting
        
        # Process the result using the agent
        answer = process_result_with_agent(result_dict, query_text)
        
        # Print the agent-generated answer
        print(f"\n--- Agent Answer for Query {query_idx} ---")
        print(answer)
        print("------------------------------")
        print(answer.content)
        
        return answer
    except Exception as e:
        print(f"Error processing result in callback: {e}")
        if future.exception():
            print(f"Future exception details: {future.exception()}")
        return None

# Test code using callback functions to process query results

if not llm:
    print("Skipping callback test due to LLM initialization failure.")
else:
    print("\n--- Starting Callback Function Test ---")
    
    # Re-create query manager if needed
    if 'query_manager' not in locals() or not hasattr(query_manager, 'is_running') or not query_manager.is_running():
        query_manager = QueryManager()
        query_manager.update_all_caches(test_onto)
        query_manager.start()
    
    # 创建一个闭包函数来捕获回调返回的answer
    def create_answer_collector():
        # 在闭包中创建一个存储结果的字典
        answers = {}
        
        # 创建一个能捕获answer的回调函数
        def answer_collector(future, query_idx, query_text):
            try:
                result_dict = future.result(timeout=5)
                # 处理结果并获取answer
                answer = process_result_with_agent(result_dict, query_text)
                # 将answer存储在闭包的answers字典中
                answers[query_idx] = answer
                print(f"查询 {query_idx} 的答案已保存")
                return answer
            except Exception as e:
                print(f"处理结果时出错: {e}")
                return None
        
        # 返回回调函数和结果字典
        return answer_collector, answers

    # 创建回调函数和结果存储字典
    callback_collector, answers = create_answer_collector()

    # 提交查询并注册回调
    callback_futures = []
    for i, query_text in enumerate(queries):
        question = revised_queries[i]
        print(f"提交查询 {i+1}: '{query_text}'")
        future = query_manager.submit_query(query_text=query_text, query_context=query_context)
        
        # 使用functools.partial创建带参数的回调函数
        from functools import partial
        callback_func = partial(callback_collector, query_idx=i+1, query_text=question)
        
        # 注册回调函数
        future.add_done_callback(callback_func)
        callback_futures.append((i+1, query_text, future))
    
    # Wait for all Futures to complete (optional but ensures all callbacks execute)
    import concurrent.futures
    import time
    
    # Non-blocking check
    all_done = False
    wait_time = 0
    max_wait_time = 120  # Maximum wait time
    check_interval = 5  # Check interval
    
    print("\nWaiting for callbacks to execute...")
    while not all_done and wait_time < max_wait_time:
        all_done = all(future[2].done() for future in callback_futures)
        if not all_done:
            print(f"Waited {wait_time} seconds, continuing to wait for callbacks...")
            time.sleep(check_interval)
            wait_time += check_interval
    
    if all_done:
        print("\nAll callbacks have completed!")
    else:
        print(f"\nTimeout waiting, some queries may not have completed. Waited {wait_time} seconds.")
    
    # Stop query manager
    print("\nStopping QueryManager...")
    query_manager.stop()
    print("QueryManager stopped.")
    
    print("--- Callback Function Test Finished ---")


--- Starting Callback Function Test ---
Class name cache updated with 1557 classes.
数据属性缓存更新完成，共 668 个属性
对象属性缓存更新完成，共 721 个属性
所有本体缓存更新完成
Dispatcher loop started on thread QueryDispatcherThread
Query Manager dispatcher started.
提交查询 1: 'Quinine definition?'
提交查询 2: 'Indicator Displacement Assay (IDA) definition?'
提交查询 3: 'What analyzes Quinine?'
提交查询 4: 'List components of Indicator Displacement Assay (IDA).'
提交查询 5: 'Find electrochemical sensors based on IDA used for Quinine detection.'
提交查询 6: 'Which hosts use host-guest recognition and are integrated with electrochemical assays?'
提交查询 7: 'Compare the stability and reproducibility properties of the Electrochemical Sensor.'
提交查询 8: 'What techniques verify the Electrochemical Sensor based on IDA?'
提交查询 9: 'Explain the sensing mechanism involving Methylene Blue (MB) displacement by Quinine from beta-Cyclodextrin (beta-CD).'
提交查询 10: 'Summarize the role of Graphene in the described electrochemical sensor.'

Waiting for callbacks to execu

In [9]:
answers

{2: "I'm sorry, I couldn't find valid information about 'What is an Indicator Displacement Assay?'.",
 1: AIMessage(content="Quinine is a naturally occurring alkaloid primarily extracted from the bark of the cinchona tree (genus *Cinchona*). It has a long history of use as an antimalarial agent, owing to its ability to interfere with the lifecycle of *Plasmodium* parasites that cause malaria. Specifically, quinine exerts its antimalarial activity by disrupting the parasite's ability to digest hemoglobin within red blood cells, which is essential for its survival and proliferation.\n\nChemically, quinine belongs to the class of cinchona alkaloids and features a complex, chiral molecular structure with multiple stereocenters. It is characterized by a quinoline core fused to a quinuclidine ring, contributing to its biological activity. Quinine is also known for its bitter taste, which has historically led to its use as a flavoring agent in tonic water, although at much lower concentration

In [12]:
from langchain_core.messages import AIMessage
for idx, answer in answers.items():
        if isinstance(answer, AIMessage):
                print(f"查询 {idx} 的最终答案: {answer.content}")
        else:
                print("error")

error
查询 1 的最终答案: Quinine is a naturally occurring alkaloid primarily extracted from the bark of the cinchona tree (genus *Cinchona*). It has a long history of use as an antimalarial agent, owing to its ability to interfere with the lifecycle of *Plasmodium* parasites that cause malaria. Specifically, quinine exerts its antimalarial activity by disrupting the parasite's ability to digest hemoglobin within red blood cells, which is essential for its survival and proliferation.

Chemically, quinine belongs to the class of cinchona alkaloids and features a complex, chiral molecular structure with multiple stereocenters. It is characterized by a quinoline core fused to a quinuclidine ring, contributing to its biological activity. Quinine is also known for its bitter taste, which has historically led to its use as a flavoring agent in tonic water, although at much lower concentrations than those used for medicinal purposes.

Beyond its antimalarial properties, quinine exhibits additional ph

In [11]:
res_list = []
for i, ques in enumerate(revised_queries):
    response = answer_llm.invoke(ques)
    res_list.append(response)
    print(f"完成 {i+1} 个回答")


完成 1 个回答
完成 2 个回答
完成 3 个回答
完成 4 个回答
完成 5 个回答
完成 6 个回答
完成 7 个回答
完成 8 个回答
完成 9 个回答
完成 10 个回答


In [13]:
for i, res in enumerate(res_list):
    print(f"查询 {i+1} 的答案是：{res.content}")

查询 1 的答案是：Quinine is a naturally occurring alkaloid best known for its antimalarial properties. Below is an overview of its origin, chemistry, pharmacology, clinical uses and safety profile.

1. Origin and History  
 • Source: Extracted from the bark of Cinchona species (“fever tree”), native to South America.  
 • Discovery: Isolated in 1820 by French chemists Pierre Joseph Pelletier and Joseph Bienaimé Caventou.  
 • Historical note: Jesuit missionaries in Peru first learned of its use to treat fevers in the 17th century.

2. Chemical Properties  
 • Formula: C20H24N2O2  
 • Structure: Quinoline core with a quinuclidine ring; chiral center yields stereoisomers.  
 • Physical form: White crystalline powder, bitter taste, poorly soluble in water.

3. Mechanism of Antimalarial Action  
 • Plasmodium attack: Interferes with parasite’s heme detoxification in red-blood-cell food vacuoles.  
 • Result: Accumulation of toxic heme kills the parasite, particularly effective against Plasmodium 

**Note on Streaming with QueryManager:**

The standard `QueryManager.submit_query()` returns a `Future` that resolves to the *final* state of the LangGraph execution. It doesn't inherently provide access to the intermediate states generated by each node.

To observe the step-by-step execution and intermediate state changes, you would typically need to interact directly with the LangGraph instance using its `stream()` method, as demonstrated in Part 2 below. Modifying the `QueryManager` to expose this stream would require significant changes to its asynchronous task handling and result reporting.

## Part 2: Direct Execution via Graph Stream

In [7]:
if not llm:
    print("Skipping Direct Graph Stream test due to LLM initialization failure.")
else:
    print("\n--- Starting Direct Graph Stream Test ---")

    # 1. Create graph instance
    print("Creating graph instance...")
    graph = create_query_graph()
    print("Graph instance created.")

    # 2. Manually create initial state dictionary
    print("Creating initial state...")
    # Use the same query as Part 1 for comparison
    # query_text_stream = "What proteins does DrugA bind to?"

    for query_text_stream in [queries[4]]:

        try:
            # Ensure we get a list of strings
            available_classes_stream = sorted([cls.name for cls in test_onto.classes() if isinstance(cls, ThingClass)])
            available_data_props_stream = sorted([dp.name for dp in test_onto.data_properties() if isinstance(dp, DataPropertyClass)])
            available_object_props_stream = sorted([op.name for op in test_onto.object_properties() if isinstance(op, ObjectPropertyClass)])
        except Exception as e:
            print(f"Error getting class names: {e}")
            available_classes_stream = []

        initial_state = {
            "query": query_text_stream,
            "source_ontology": test_ontology_settings, # Pass the actual ontology object
            "available_classes": available_classes_stream,
            "available_data_properties": available_data_props_stream,
            "available_object_properties": available_object_props_stream,
            "query_type": "information_retrieval",
            "query_strategy": None,
            "originating_team": "test_notebook_stream",
            "originating_stage": "manual_stream_test",
            "query_results": {},
            "normalized_query": None,
            "execution_plan": None,
            "validation_report": None,
            "sparql_query": None,
            "status": "initialized",
            "stage": "initialized",
            "previous_stage": None,
            "error": None,
            "messages": [] # LangGraph expects messages field
        }
        print("Initial state prepared.")
        # print(json.dumps(initial_state, indent=2, default=str)) # Optionally print initial state (ontology won't serialize well)

        # 3. Execute and iterate stream
        print("\n--- Streaming Graph Execution --- ")
        try:
            stream_counter = 0
            # Use stream method to get intermediate steps
            for chunk in graph.stream(initial_state):
                stream_counter += 1
                print(f"\n--- Chunk {stream_counter} --- ")
                # Chunks are dictionaries where keys are node names that just ran
                # and values are the outputs (state updates) returned by that node
                # Use default=str to handle potential non-serializable objects in the state
                print(json.dumps(chunk, indent=2, default=str))
                print("-" * 30)
            print("\n--- Graph Stream Finished --- ")
        except Exception as e:
            print(f"\nError during graph stream: {e}")
            import traceback
            traceback.print_exc() # Print full traceback for stream errors
        
        print(f"query:{query_text_stream} has been finished.")

    print("--- Direct Graph Stream Test Finished ---")


--- Starting Direct Graph Stream Test ---
Creating graph instance...
Graph instance created.
Creating initial state...
Initial state prepared.

--- Streaming Graph Execution --- 
[('system', 'You are an expert ontology query parser. Your task is to convert natural language queries into a structured format.\n1. Strictly adhere to the NormalizedQuery JSON schema for the output.\n2. Refer to the provided list of available ontology classes to identify entities.\n3. Refer to the provided lists of data properties and object properties to identify property relationships.\n4. Note that there are SourcedInformation objects that provide additional metadata. When queries involve concepts like "source", "description", or "definition", consider that these information are not related to relations.'), ('user', "Available classes: 1,1-butane(1,4-diyl)bis(2-aminopyridine)_bromide(DPAD), 1_h_nmr, 1_h_nmr_spectrum, 1_hnmr_spectra, 1h_nmr, 1h_nmr_spectrum, 1h_nuclear_magnetic_resonance_spectroscopy(1H_NM

In [8]:
print(initial_state["normalized_query"])

None


# QueryManager 检查


In [7]:
import threading
import traceback
import sys

def check_threads():
    """检查当前进程中的活跃线程"""
    print(f"当前活跃线程数: {threading.active_count()}")
    
    print("\n当前活跃线程:")
    for t in threading.enumerate():
        print(f"- {t.name} (daemon: {t.daemon}, 活动: {t.is_alive()})")
    
    print("\n线程调用栈:")
    query_manager_threads = []
    for thread_id, frame in sys._current_frames().items():
        thread_name = "Unknown"
        for t in threading.enumerate():
            if t.ident == thread_id:
                thread_name = t.name
                break
        
        # 检查是否是QueryManager相关线程
        is_query_thread = False
        stack_trace = traceback.extract_stack(frame)
        for filename, _, _, _ in stack_trace:
            if "query_manager" in filename or "ThreadPool" in filename:
                is_query_thread = True
                query_manager_threads.append(thread_name)
                break
        
        print(f"线程ID: {thread_id}, 名称: {thread_name}{' (QueryManager相关)' if is_query_thread else ''}")
        for filename, lineno, name, line in stack_trace[-10:]:  # 只显示最近10个调用
            print(f"  文件: {filename.split('/')[-1]}, 行: {lineno}, 函数: {name}")
            if line:
                print(f"    代码: {line}")
        print("")
    
    if query_manager_threads:
        print(f"\n发现 {len(query_manager_threads)} 个QueryManager相关线程: {', '.join(query_manager_threads)}")
    else:
        print("\n未发现QueryManager相关线程")

# 执行检查
check_threads()

当前活跃线程数: 6

当前活跃线程:
- MainThread (daemon: False, 活动: True)
- IOPub (daemon: True, 活动: True)
- Heartbeat (daemon: True, 活动: True)
- Control (daemon: True, 活动: True)
- IPythonHistorySavingThread (daemon: True, 活动: True)
- Thread-1 (daemon: True, 活动: True)

线程调用栈:
线程ID: 13768, 名称: Thread-1
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1012, 函数: _bootstrap
    代码: self._bootstrap_inner()
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1041, 函数: _bootstrap_inner
    代码: self.run()
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\site-packages\ipykernel\parentpoller.py, 行: 93, 函数: run
    代码: result = ctypes.windll.kernel32.WaitForMultipleObjects(  # type:ignore[attr-defined]

线程ID: 33716, 名称: IPythonHistorySavingThread
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1012, 函数: _bootstrap
    代码: self._bootstrap_inner()
  文件: d:\AnacondaEnPs\envs\OntologyConstruction\Lib\threading.py, 行: 1041, 函数: _bootstrap_inner
    代码: sel

In [8]:
# 查看缓存内容（如果有缓存的查询）
cache_content = query_manager.query_queue_manager.cache.cache
print(f"缓存中的查询数量: {len(cache_content)}")

# 查看缓存的时间戳信息
timestamps = query_manager.query_queue_manager.cache.timestamps
if timestamps:
    print("\n缓存时间戳:")
    for key, timestamp in timestamps.items():
        print(f"查询: {key[:50]}... - 时间: {timestamp}")
        # 计算剩余有效时间
        ttl = query_manager.query_queue_manager.cache.ttl  # 默认3600秒（1小时）
        from datetime import datetime, timedelta
        remaining = timestamp + timedelta(seconds=ttl) - datetime.now()
        print(f"  剩余有效时间: {remaining}")

# 如果需要手动清除缓存
# query_manager.query_queue_manager.cache.clear()
# print("缓存已清除")

缓存中的查询数量: 0
